# SimpleTM — Multivariate Time Series Forecasting

## BTP Project: Comparing SWT vs FFT Tokenization

This notebook runs all 3 model variants:
1. **SimpleTM** — Original (SWT tokenization + Geometric Attention)
2. **SimpleTM_SWT** — SWT-only baseline
3. **SimpleTM_FFT** — FFT tokenization + Geometric Attention

---

## 1. Setup & Install Dependencies

In [1]:
!pip install PyWavelets gdown --quiet

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4


In [2]:
# ======================================================================
# Patch for NumPy 2.0 compatibility (np.Inf → np.inf)
# ======================================================================
import subprocess
import os
REPO_DIR = "/kaggle/working/SimpleTM"
# After cloning, patch the file
def patch_numpy_compat(repo_dir):
    """Fix np.Inf → np.inf for NumPy 2.0+ compatibility."""
    tools_path = os.path.join(repo_dir, "utils/tools.py")
    if os.path.exists(tools_path):
        with open(tools_path, 'r') as f:
            content = f.read()
        content = content.replace('np.Inf', 'np.inf')
        with open(tools_path, 'w') as f:
            f.write(content)
        print("✓ Patched utils/tools.py: np.Inf → np.inf")
    else:
        print(f"⚠ {tools_path} not found yet — run this after cloning")

## 2. Clone Repository

In [3]:
import os

# Clone the repo (update URL to your fork if needed)
REPO_URL = "https://github.com/agamswami/SimpleTMG.git"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned.")
else:
    print("Repository already exists.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

Cloning into '/kaggle/working/SimpleTM'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 57 (delta 12), reused 57 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 7.08 MiB | 27.90 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Repository cloned.
Working directory: /kaggle/working/SimpleTM
total 5728
drwxr-xr-x 10 root root    4096 Mar 10 07:02  .
drwxr-xr-x  4 root root    4096 Mar 10 07:02  ..
-rw-r--r--  1 root root 5093822 Mar 10 07:02  3089_SimpleTM_A_Simple_Baselin.pdf
-rw-r--r--  1 root root  233348 Mar 10 07:02  btp_context.pdf
drwxr-xr-x  2 root root    4096 Mar 10 07:02  data_provider
-rw-r--r--  1 root root     301 Mar 10 07:02  Dockerfile
-rw-r--r--  1 root root     268 Mar 10 07:02  environment.yml
drwxr-xr-x  2 root root    4096 Mar 10 07:02  experiments
drwxr-xr-x  2 root root    4096 Mar 10 07:02  figures
drwxr-xr-x  8 root root    4096 Mar 10 07:0

In [4]:
!sed -i 's/np\.Inf/np.inf/g' /kaggle/working/SimpleTM/utils/tools.py


## 2b. Copy Extended Model Files

If you uploaded the extended files (SimpleTM_SWT.py, SimpleTM_FFT.py, FFTAttention_Family.py) to Kaggle as a dataset, copy them here. Otherwise, create them inline below.

In [5]:
# # ======================================================================
# # FFT Attention Layer — layers/FFTAttention_Family.py
# # ======================================================================

# fft_attention_code = '''
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from math import sqrt
# from layers.SWTAttention_Family import GeomAttention


# class FFTEmbedding(nn.Module):
#     """
#     FFT-based tokenization to replace SWT.
#     Decomposes input signal into m+1 frequency bands.
#     """
#     def __init__(self, d_channel=16, decompose=True, m=2, seq_len=None):
#         super().__init__()
#         self.decompose = decompose
#         self.d_channel = d_channel
#         self.m = m
#         self.band_weights = nn.Parameter(torch.ones(m + 1), requires_grad=True)

#     def forward(self, x):
#         if self.decompose:
#             return self.fft_decomposition(x)
#         else:
#             return self.fft_reconstruction(x)

#     def fft_decomposition(self, x):
#         B, N, L = x.shape
#         X_fft = torch.fft.rfft(x, dim=-1)
#         n_freqs = X_fft.shape[-1]
#         band_size = max(1, n_freqs // (self.m + 1))
#         coeffs = []
#         for i in range(self.m + 1):
#             mask = torch.zeros(n_freqs, device=x.device, dtype=torch.float32)
#             start = i * band_size
#             end = min((i + 1) * band_size, n_freqs) if i < self.m else n_freqs
#             mask[start:end] = 1.0
#             X_band = X_fft * mask.unsqueeze(0).unsqueeze(0)
#             band_signal = torch.fft.irfft(X_band, n=L, dim=-1)
#             band_signal = band_signal * self.band_weights[i]
#             coeffs.append(band_signal)
#         return torch.stack(coeffs, dim=-2)

#     def fft_reconstruction(self, coeffs):
#         inv_weights = 1.0 / (self.band_weights + 1e-8)
#         weighted_coeffs = coeffs * inv_weights.unsqueeze(0).unsqueeze(0).unsqueeze(-1)
#         reconstructed = weighted_coeffs.sum(dim=-2)
#         return reconstructed


# class FFTGeomAttentionLayer(nn.Module):
#     """
#     Attention layer using FFT tokenization with Geometric Attention.
#     Drop-in replacement for GeomAttentionLayer.
#     """
#     def __init__(self, attention, d_model,
#                  m=2, d_channel=None, geomattn_dropout=0.5):
#         super(FFTGeomAttentionLayer, self).__init__()
#         self.d_channel = d_channel
#         self.inner_attention = attention
#         self.fft_decompose = FFTEmbedding(d_channel=self.d_channel, decompose=True, m=m)
#         self.query_projection = nn.Sequential(
#             nn.Linear(d_model, d_model), nn.Dropout(geomattn_dropout))
#         self.key_projection = nn.Sequential(
#             nn.Linear(d_model, d_model), nn.Dropout(geomattn_dropout))
#         self.value_projection = nn.Sequential(
#             nn.Linear(d_model, d_model), nn.Dropout(geomattn_dropout))
#         self.out_projection = nn.Sequential(
#             nn.Linear(d_model, d_model),
#             FFTEmbedding(d_channel=self.d_channel, decompose=False, m=m),
#         )

#     def forward(self, queries, keys, values, attn_mask=None, tau=None, delta=None):
#         queries = self.fft_decompose(queries)
#         keys = self.fft_decompose(keys)
#         values = self.fft_decompose(values)
#         queries = self.query_projection(queries).permute(0,3,2,1)
#         keys = self.key_projection(keys).permute(0,3,2,1)
#         values = self.value_projection(values).permute(0,3,2,1)
#         out, attn = self.inner_attention(queries, keys, values)
#         out = self.out_projection(out.permute(0,3,2,1))
#         return out, attn
# '''

# os.makedirs('layers', exist_ok=True)
# with open('layers/FFTAttention_Family.py', 'w') as f:
#     f.write(fft_attention_code)
# print("Created layers/FFTAttention_Family.py")

In [6]:
# # ======================================================================
# # SimpleTM_SWT Model — model/SimpleTM_SWT.py
# # ======================================================================

# swt_model_code = '''
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from layers.Transformer_Encoder import Encoder, EncoderLayer
# from layers.SWTAttention_Family import GeomAttentionLayer, GeomAttention
# from layers.Embed import DataEmbedding_inverted

# class Model(nn.Module):
#     """SimpleTM with SWT-Only Tokenization (Original Baseline)."""
#     def __init__(self, configs):
#         super(Model, self).__init__()
#         self.seq_len = configs.seq_len
#         self.pred_len = configs.pred_len
#         self.output_attention = configs.output_attention
#         self.use_norm = configs.use_norm
#         self.geomattn_dropout = configs.geomattn_dropout
#         self.alpha = configs.alpha
#         self.kernel_size = configs.kernel_size
#         enc_embedding = DataEmbedding_inverted(configs.seq_len, configs.d_model,
#                                                configs.embed, configs.freq, configs.dropout)
#         self.enc_embedding = enc_embedding
#         encoder = Encoder(
#             [EncoderLayer(
#                 GeomAttentionLayer(
#                     GeomAttention(False, configs.factor, attention_dropout=configs.dropout,
#                                   output_attention=configs.output_attention, alpha=self.alpha),
#                     configs.d_model, requires_grad=configs.requires_grad, wv=configs.wv,
#                     m=configs.m, d_channel=configs.dec_in, kernel_size=self.kernel_size,
#                     geomattn_dropout=self.geomattn_dropout),
#                 configs.d_model, configs.d_ff,
#                 dropout=configs.dropout, activation=configs.activation,
#             ) for l in range(configs.e_layers)],
#             norm_layer=torch.nn.LayerNorm(configs.d_model))
#         self.encoder = encoder
#         projector = nn.Linear(configs.d_model, self.pred_len, bias=True)
#         self.projector = projector

#     def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
#         if self.use_norm:
#             means = x_enc.mean(1, keepdim=True).detach()
#             x_enc = x_enc - means
#             stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
#             x_enc = x_enc / stdev
#         _, _, N = x_enc.shape
#         enc_out = self.enc_embedding(x_enc, x_mark_enc)
#         enc_out, attns = self.encoder(enc_out, attn_mask=None)
#         dec_out = self.projector(enc_out).permute(0, 2, 1)[:, :, :N]
#         if self.use_norm:
#             dec_out = dec_out * (stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))
#             dec_out = dec_out + (means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))
#         return dec_out, attns

#     def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
#         dec_out, attns = self.forecast(x_enc, None, None, None)
#         return dec_out, attns
# '''

# os.makedirs('model', exist_ok=True)
# with open('model/SimpleTM_SWT.py', 'w') as f:
#     f.write(swt_model_code)
# print("Created model/SimpleTM_SWT.py")

In [7]:
# # ======================================================================
# # SimpleTM_FFT Model — model/SimpleTM_FFT.py
# # ======================================================================

# fft_model_code = '''
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from layers.Transformer_Encoder import Encoder, EncoderLayer
# from layers.FFTAttention_Family import FFTGeomAttentionLayer
# from layers.SWTAttention_Family import GeomAttention
# from layers.Embed import DataEmbedding_inverted

# class Model(nn.Module):
#     """SimpleTM with FFT-Only Tokenization."""
#     def __init__(self, configs):
#         super(Model, self).__init__()
#         self.seq_len = configs.seq_len
#         self.pred_len = configs.pred_len
#         self.output_attention = configs.output_attention
#         self.use_norm = configs.use_norm
#         self.geomattn_dropout = configs.geomattn_dropout
#         self.alpha = configs.alpha
#         enc_embedding = DataEmbedding_inverted(configs.seq_len, configs.d_model,
#                                                configs.embed, configs.freq, configs.dropout)
#         self.enc_embedding = enc_embedding
#         encoder = Encoder(
#             [EncoderLayer(
#                 FFTGeomAttentionLayer(
#                     GeomAttention(False, configs.factor, attention_dropout=configs.dropout,
#                                   output_attention=configs.output_attention, alpha=self.alpha),
#                     configs.d_model, m=configs.m, d_channel=configs.dec_in,
#                     geomattn_dropout=self.geomattn_dropout),
#                 configs.d_model, configs.d_ff,
#                 dropout=configs.dropout, activation=configs.activation,
#             ) for l in range(configs.e_layers)],
#             norm_layer=torch.nn.LayerNorm(configs.d_model))
#         self.encoder = encoder
#         projector = nn.Linear(configs.d_model, self.pred_len, bias=True)
#         self.projector = projector

#     def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
#         if self.use_norm:
#             means = x_enc.mean(1, keepdim=True).detach()
#             x_enc = x_enc - means
#             stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
#             x_enc = x_enc / stdev
#         _, _, N = x_enc.shape
#         enc_out = self.enc_embedding(x_enc, x_mark_enc)
#         enc_out, attns = self.encoder(enc_out, attn_mask=None)
#         dec_out = self.projector(enc_out).permute(0, 2, 1)[:, :, :N]
#         if self.use_norm:
#             dec_out = dec_out * (stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))
#             dec_out = dec_out + (means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))
#         return dec_out, attns

#     def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
#         dec_out, attns = self.forecast(x_enc, None, None, None)
#         return dec_out, attns
# '''

# with open('model/SimpleTM_FFT.py', 'w') as f:
#     f.write(fft_model_code)
# print("Created model/SimpleTM_FFT.py")

In [8]:
# # ======================================================================
# # Patch exp_basic.py to register new models
# # ======================================================================

# exp_basic_path = 'experiments/exp_basic.py'
# with open(exp_basic_path, 'r') as f:
#     content = f.read()

# # Add imports
# if 'SimpleTM_SWT' not in content:
#     content = content.replace(
#         'from model import SimpleTM',
#         'from model import SimpleTM\nfrom model import SimpleTM_SWT\nfrom model import SimpleTM_FFT'
#     )
#     content = content.replace(
#         "'SimpleTM': SimpleTM,\n        }",
#         "'SimpleTM': SimpleTM,\n            'SimpleTM_SWT': SimpleTM_SWT,\n            'SimpleTM_FFT': SimpleTM_FFT,\n        }"
#     )
#     with open(exp_basic_path, 'w') as f:
#         f.write(content)
#     print("Patched exp_basic.py")
# else:
#     print("exp_basic.py already patched")

## 3. Download Dataset

Download the dataset from Google Drive using `gdown`.

In [9]:
!pip install gdown --quiet
import gdown
import zipfile

# Google Drive file ID
FILE_ID = "1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    # Download
    url = f"https://drive.google.com/uc?id={FILE_ID}"
    gdown.download(url, OUTPUT_PATH, quiet=False)
    
    # Extract
    os.makedirs(DATASET_DIR, exist_ok=True)
    if OUTPUT_PATH.endswith('.zip'):
        with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
            zip_ref.extractall(DATASET_DIR)
        print(f"Extracted dataset to {DATASET_DIR}")
    else:
        # If it's not a zip, it might be a single file — move it
        import shutil
        shutil.move(OUTPUT_PATH, os.path.join(DATASET_DIR, 'data.csv'))
        print(f"Moved dataset to {DATASET_DIR}")
else:
    print(f"Dataset directory already exists: {DATASET_DIR}")

# List dataset contents
for root, dirs, files in os.walk(DATASET_DIR):
    level = root.replace(DATASET_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files[:10]:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f'{subindent}{file} ({size_mb:.2f} MB)')

Downloading...
From (original): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx
From (redirected): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx&confirm=t&uuid=887b3826-c1c5-4d24-a0df-74d1a85e1e4d
To: /kaggle/working/dataset.zip
100%|██████████| 171M/171M [00:02<00:00, 81.5MB/s] 


Extracted dataset to ./dataset
dataset/
  __MACOSX/
    ._SimpleTM_datasets (0.00 MB)
    SimpleTM_datasets/
      ._PEMS (0.00 MB)
      ._Solar (0.00 MB)
      ._electricity (0.00 MB)
      ._.DS_Store (0.00 MB)
      ._weather (0.00 MB)
      ._ETT-small (0.00 MB)
      ._traffic (0.00 MB)
      Solar/
        ._solar_AL.txt (0.00 MB)
      traffic/
        ._.DS_Store (0.00 MB)
        ._traffic.csv (0.00 MB)
      PEMS/
        ._PEMS03.npz (0.00 MB)
        ._PEMS07.npz (0.00 MB)
        ._PEMS04.npz (0.00 MB)
        ._PEMS08.npz (0.00 MB)
      ETT-small/
        ._ETTh1.csv (0.00 MB)
        ._ETTh2.csv (0.00 MB)
        ._ETTm1.csv (0.00 MB)
        ._ETTm2.csv (0.00 MB)
      weather/
        ._weather.csv (0.00 MB)
      electricity/
        ._electricity.csv (0.00 MB)
        ._.DS_Store (0.00 MB)
  SimpleTM_datasets/
    .DS_Store (0.01 MB)
    Solar/
      solar_AL.txt (171.68 MB)
    traffic/
      .DS_Store (0.01 MB)
      traffic.csv (130.16 MB)
    PEMS/
      PEMS04

## 4. Configuration

Set up common experiment parameters. **Adjust `ROOT_PATH` and `DATA_PATH` based on your dataset structure.**

In [10]:
# ======================================================================
# CONFIGURE THESE BASED ON YOUR DATASET
# ======================================================================

# For ETTh1 dataset:
ROOT_PATH = "/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small"  # Adjust this path!
DATA_PATH = "ETTh1.csv"             # Adjust this filename!
DATA_TYPE = "ETTh1"                 # Options: ETTh1, ETTh2, ETTm1, ETTm2, custom
ENC_IN = 7                          # Number of variates
DEC_IN = 7
C_OUT = 7

# Common hyperparameters
SEQ_LEN = 96         # Input sequence length
PRED_LEN = 96        # Prediction horizon
D_MODEL = 32         # Token dimension
D_FF = 32            # Feed-forward dimension
E_LAYERS = 1         # Number of encoder layers
M = 3                # Decomposition levels (SWT/FFT bands)
ALPHA = 1.0          # Geometric attention alpha
LR = 0.0001          # Learning rate
BATCH_SIZE = 32      # Batch size
TRAIN_EPOCHS = 10    # Training epochs
PATIENCE = 3         # Early stopping patience

print(f"Dataset: {DATA_TYPE} ({DATA_PATH})")
print(f"Variates: {ENC_IN}, Seq: {SEQ_LEN}, Pred: {PRED_LEN}")
print(f"Model: d_model={D_MODEL}, d_ff={D_FF}, layers={E_LAYERS}")

Dataset: ETTh1 (ETTh1.csv)
Variates: 7, Seq: 96, Pred: 96
Model: d_model=32, d_ff=32, layers=1


## 5. Train Original SimpleTM (SWT + Geometric Attention)

In [11]:
!python -u run.py \
  --is_training 1 \
  --model SimpleTM \
  --model_id {DATA_TYPE}_original \
  --data {DATA_TYPE} \
  --root_path {ROOT_PATH} \
  --data_path {DATA_PATH} \
  --features M \
  --seq_len {SEQ_LEN} \
  --pred_len {PRED_LEN} \
  --e_layers {E_LAYERS} \
  --d_model {D_MODEL} \
  --d_ff {D_FF} \
  --enc_in {ENC_IN} \
  --dec_in {DEC_IN} \
  --c_out {C_OUT} \
  --wv db1 \
  --m {M} \
  --alpha {ALPHA} \
  --learning_rate {LR} \
  --batch_size {BATCH_SIZE} \
  --train_epochs {TRAIN_EPOCHS} \
  --patience {PATIENCE} \
  --num_workers 2 \
  --fix_seed 2025

Args in experiment:
Namespace(is_training=1, model_id='ETTh1_original', model='SimpleTM', data='ETTh1', root_path='/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0, requires_grad=True, wv='db1', m=3, kernel_size=None, alpha=1.0, l1_wei

## 6. Train SimpleTM_SWT (SWT-Only Baseline)

In [12]:
!python -u run.py \
  --is_training 1 \
  --model SimpleTM_SWT \
  --model_id {DATA_TYPE}_SWT \
  --data {DATA_TYPE} \
  --root_path {ROOT_PATH} \
  --data_path {DATA_PATH} \
  --features M \
  --seq_len {SEQ_LEN} \
  --pred_len {PRED_LEN} \
  --e_layers {E_LAYERS} \
  --d_model {D_MODEL} \
  --d_ff {D_FF} \
  --enc_in {ENC_IN} \
  --dec_in {DEC_IN} \
  --c_out {C_OUT} \
  --wv db1 \
  --m {M} \
  --alpha {ALPHA} \
  --learning_rate {LR} \
  --batch_size {BATCH_SIZE} \
  --train_epochs {TRAIN_EPOCHS} \
  --patience {PATIENCE} \
  --num_workers 2 \
  --fix_seed 2025

Args in experiment:
Namespace(is_training=1, model_id='ETTh1_SWT', model='SimpleTM_SWT', data='ETTh1', root_path='/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0, requires_grad=True, wv='db1', m=3, kernel_size=None, alpha=1.0, l1_weig

## 7. Train SimpleTM_FFT (FFT-Only Tokenization)

In [13]:
!python -u run.py \
  --is_training 1 \
  --model SimpleTM_FFT \
  --model_id {DATA_TYPE}_FFT \
  --data {DATA_TYPE} \
  --root_path {ROOT_PATH} \
  --data_path {DATA_PATH} \
  --features M \
  --seq_len {SEQ_LEN} \
  --pred_len {PRED_LEN} \
  --e_layers {E_LAYERS} \
  --d_model {D_MODEL} \
  --d_ff {D_FF} \
  --enc_in {ENC_IN} \
  --dec_in {DEC_IN} \
  --c_out {C_OUT} \
  --wv db1 \
  --m {M} \
  --alpha {ALPHA} \
  --learning_rate {LR} \
  --batch_size {BATCH_SIZE} \
  --train_epochs {TRAIN_EPOCHS} \
  --patience {PATIENCE} \
  --num_workers 2 \
  --fix_seed 2025

Args in experiment:
Namespace(is_training=1, model_id='ETTh1_FFT', model='SimpleTM_FFT', data='ETTh1', root_path='/kaggle/working/SimpleTM/dataset/SimpleTM_datasets/ETT-small', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficient_training=False, use_norm=True, partial_start_index=0, requires_grad=True, wv='db1', m=3, kernel_size=None, alpha=1.0, l1_weig

## 8. Compare Results

Parse the results file and display a comparison table.

In [14]:
import re
import pandas as pd

results_file = "result_long_term_forecast.txt"

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        content = f.read()
    
    # Parse results
    entries = content.strip().split('\n\n')
    results = []
    
    for entry in entries:
        lines = entry.strip().split('\n')
        if len(lines) >= 2:
            setting = lines[0].strip()
            metrics_line = lines[1].strip()
            
            # Extract model type from setting
            if '_FFT_' in setting or setting.startswith('FFT'):
                model_type = 'SimpleTM_FFT'
            elif '_SWT_' in setting or setting.startswith('SWT'):
                model_type = 'SimpleTM_SWT'
            else:
                model_type = 'SimpleTM (Original)'
            
            # Parse MSE and MAE
            mse_match = re.search(r'mse:([\d.]+)', metrics_line)
            mae_match = re.search(r'mae:([\d.]+)', metrics_line)
            
            if mse_match and mae_match:
                results.append({
                    'Model': model_type,
                    'Setting': setting[:60] + '...' if len(setting) > 60 else setting,
                    'MSE': float(mse_match.group(1)),
                    'MAE': float(mae_match.group(1)),
                })
    
    if results:
        df = pd.DataFrame(results)
        print("\n" + "="*70)
        print("                    RESULTS COMPARISON")
        print("="*70)
        print(df.to_string(index=False))
        print("="*70)
        
        # Summary by model type
        print("\n--- Average Metrics by Model ---")
        summary = df.groupby('Model')[['MSE', 'MAE']].mean()
        print(summary.to_string())
    else:
        print("No parseable results found.")
else:
    print(f"Results file '{results_file}' not found. Run training first!")


                    RESULTS COMPARISON
              Model                                                         Setting      MSE      MAE
SimpleTM (Original) ETTh1_original_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.00... 0.451259 0.446162
       SimpleTM_SWT ETTh1_SWT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_ty... 0.451259 0.446162
       SimpleTM_FFT ETTh1_FFT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_ty... 0.457091 0.449469

--- Average Metrics by Model ---
                          MSE       MAE
Model                                  
SimpleTM (Original)  0.451259  0.446162
SimpleTM_FFT         0.457091  0.449469
SimpleTM_SWT         0.451259  0.446162


## 9. Visualize Predictions

Display saved prediction vs ground-truth plots.

In [15]:
import glob
from IPython.display import Image, display
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

checkpoint_dirs = sorted(glob.glob('./checkpoints/*'))
print(f"Found {len(checkpoint_dirs)} checkpoint directories:")
for d in checkpoint_dirs:
    pdfs = glob.glob(os.path.join(d, '*.pdf'))
    print(f"  {os.path.basename(d)}: {len(pdfs)} visualization PDFs")

# Display first visualization from each model
for d in checkpoint_dirs:
    pdfs = sorted(glob.glob(os.path.join(d, '*.pdf')))
    if pdfs:
        print(f"\n--- {os.path.basename(d)} ---")
        print(f"First visualization: {pdfs[0]}")

Found 3 checkpoint directories:
  ETTh1_FFT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True: 5 visualization PDFs
  ETTh1_SWT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True: 5 visualization PDFs
  ETTh1_original_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True: 5 visualization PDFs

--- ETTh1_FFT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True ---
First visualization: ./checkpoints/ETTh1_FFT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True/0.pdf

--- ETTh1_SWT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True ---
First visualization: ./checkpoints/ETTh1_SWT_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True/0.pdf

--- ETTh1_original_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True ---
First visualization: ./checkpoints/ETTh1_original_ETTh1_96_96_32_32_1_db1_None_3_1.0_5e-05_0.0001_type1_32_2025_True/0.pdf
